In [1]:
import pandas as pd
import re
import gensim
import gensim.downloader as api
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt')  # Download tokenizer if not already installed

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Download stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Load dataset
df = pd.read_csv(r'F:\pbl research\5th opetion\CrisisLexT6\combined final.csv',encoding='utf-8')
#df = pd.read_csv("your_dataset.csv")  # Change to your file path

# Preprocessing function
def clean_text(tweet):
    tweet = re.sub(r"http\S+|www\S+|https\S+", '', tweet, flags=re.MULTILINE)  # Remove URLs
    tweet = re.sub(r'\@\w+|\#', '', tweet)  # Remove mentions and hashtags
    tweet = tweet.lower()  # Convert to lowercase
    tweet = re.sub(r'[^\w\s]', '', tweet)  # Remove punctuation
    tweet = " ".join([word for word in tweet.split() if word not in stop_words])  # Remove stopwords
    return tweet

# Apply text cleaning
df['cleaned_text'] = df['tweet'].apply(clean_text)

# Convert text to numerical representation (BoW)
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df['cleaned_text'])  # Feature matrix
y = df['label']  # Target variable (on-topic/off-topic)
# Train Word2Vec model
word2vec_model = Word2Vec(sentences=df["cleaned_text"], vector_size=100, window=5, min_count=2, workers=4)
import numpy as np

def get_tweet_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

df["word2vec_features"] = df["cleaned_text"].apply(lambda x: get_tweet_vector(x, word2vec_model))
# Convert list of arrays into a numpy matrix
X = np.vstack(df["word2vec_features"])
y = df["label"]  # Your target labels (e.g., 'on-topic' or 'off-topic')

# Split dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Check class distribution after split
print("Training set class distribution:\n", y_train.value_counts())
print("Testing set class distribution:\n", y_test.value_counts())

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Charvi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Charvi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Training set class distribution:
 label
on-topic     25986
off-topic    22079
Name: count, dtype: int64
Testing set class distribution:
 label
on-topic     6476
off-topic    5541
Name: count, dtype: int64


In [2]:
# Train Logistic Regression model
print("📌 Logistic Regression Results:")
model = LogisticRegression(C=0.3)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
print("Actual iterations used:", model.n_iter_)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred))

train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")



📌 Logistic Regression Results:
Actual iterations used: [38]
Accuracy: 0.7329
Classification Report:
               precision    recall  f1-score   support

   off-topic       0.75      0.64      0.69      5541
    on-topic       0.72      0.82      0.77      6476

    accuracy                           0.73     12017
   macro avg       0.74      0.73      0.73     12017
weighted avg       0.73      0.73      0.73     12017

Training Accuracy: 0.7266
Testing Accuracy: 0.7329


In [2]:
#svm
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Initialize SVM model with a linear kernel
svm_model = SVC(kernel='linear', random_state=42)

# Train the model
svm_model.fit(X_train, y_train)

# Predict on training and test data
y_train_pred_svm = svm_model.predict(X_train)
y_test_pred_svm = svm_model.predict(X_test)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred_svm)
test_accuracy = accuracy_score(y_test, y_test_pred_svm)

# Print results
print("📌 Support Vector Machine (SVM) Results:")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print("\nClassification Report on Test Data:")
print(classification_report(y_test, y_test_pred_svm))



📌 Support Vector Machine (SVM) Results:
Training Accuracy: 0.7363
Testing Accuracy: 0.7375

Classification Report on Test Data:
              precision    recall  f1-score   support

   off-topic       0.76      0.63      0.69      5541
    on-topic       0.72      0.83      0.77      6476

    accuracy                           0.74     12017
   macro avg       0.74      0.73      0.73     12017
weighted avg       0.74      0.74      0.73     12017



In [3]:
#random forset
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize and train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict on test data
y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

# Calculate accuracy
train_accuracy = accuracy_score(y_train,y_train_pred_rf)
test_accuracy = accuracy_score(y_test, y_test_pred_rf)

# Evaluate model
print("📌 Random Forest Results:")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print(classification_report(y_test, y_test_pred_rf))


📌 Random Forest Results:
Training Accuracy: 0.9976
Testing Accuracy: 0.8312
              precision    recall  f1-score   support

   off-topic       0.81      0.82      0.82      5541
    on-topic       0.85      0.84      0.84      6476

    accuracy                           0.83     12017
   macro avg       0.83      0.83      0.83     12017
weighted avg       0.83      0.83      0.83     12017



In [4]:
#decision trees
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize Decision Tree classifier
dt_model = DecisionTreeClassifier(max_depth=20, random_state=42)

# Train the model
dt_model.fit(X_train, y_train)

# Predictions
y_train_pred = dt_model.predict(X_train)
y_test_pred = dt_model.predict(X_test)

# Calculate Accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Print results
print("📌 Decision Tree Results:")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print("\nClassification Report on Test Data:")
print(classification_report(y_test, y_test_pred))


📌 Decision Tree Results:
Training Accuracy: 0.9530
Testing Accuracy: 0.7406

Classification Report on Test Data:
              precision    recall  f1-score   support

   off-topic       0.74      0.67      0.70      5541
    on-topic       0.74      0.80      0.77      6476

    accuracy                           0.74     12017
   macro avg       0.74      0.74      0.74     12017
weighted avg       0.74      0.74      0.74     12017



In [5]:
#kNN
from sklearn.neighbors import KNeighborsClassifier

# Initialize KNN classifier with 5 neighbors
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train the model
knn_model.fit(X_train, y_train)

# Predictions
y_train_pred = knn_model.predict(X_train)
y_test_pred = knn_model.predict(X_test)

# Calculate Accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Print results
print("📌 KNN Results:")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print("\nClassification Report on Test Data:")
print(classification_report(y_test, y_test_pred))

📌 KNN Results:
Training Accuracy: 0.8250
Testing Accuracy: 0.7543

Classification Report on Test Data:
              precision    recall  f1-score   support

   off-topic       0.85      0.56      0.68      5541
    on-topic       0.71      0.92      0.80      6476

    accuracy                           0.75     12017
   macro avg       0.78      0.74      0.74     12017
weighted avg       0.78      0.75      0.74     12017

